# 03. Build evidence variations and split train/validation

Run after `02_web_search_tavily.py`. For every claim, builds three evidence variations (gold, web search, gold + web search) and splits claims 90/10 into training and validation.

Input: `../data/training/sample_tavily.jsonl`.

Outputs: `../data/training/variations.jsonl` and `../data/training/{train,validation}_{gold,web_search,gold_plus_web_search}.jsonl`.


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "../data/training/"

In [ ]:
df_sample = pd.read_json(os.path.join(OUTPUT_DIR, "sample_tavily.jsonl"), lines=True)

In [ ]:
print(df_sample.shape)
df_sample.tail()

In [ ]:
# Create three evidence variations for the sampled training pool
print(f"Sample shape: {df_sample.shape}")
print(f"Columns: {list(df_sample.columns)}")

# Define columns to keep in final output
columns_to_keep = ['claim', 'evidence', 'label', 'evidence_type', 'dataset']

# Create variations list
variations = []
# Variation 1: Gold evidence only (original evidence column)
df_gold = df_sample[['claim', 'evidence', 'label', 'dataset']].copy()
df_gold['evidence_type'] = 'gold'
variations.append(df_gold)

# Variation 2: Web search evidence only (evidence_tavily_5_results column)
df_web = df_sample[['claim', 'label', 'dataset']].copy()
df_web['evidence'] = df_sample['evidence_tavily_5_results']
df_web['evidence_type'] = 'web_search'
variations.append(df_web)

# Variation 3: Combined evidence (gold + web search)
df_combined = df_sample[['claim', 'label', 'dataset']].copy()
df_combined['evidence'] = (
    df_sample['evidence'].astype(str) + 
    '\n\n' + 
    df_sample['evidence_tavily_5_results'].astype(str)
)
df_combined['evidence_type'] = 'gold_plus_web_search'
variations.append(df_combined)

# Filter columns for each variation
df_gold_filtered = df_gold[columns_to_keep]
df_web_filtered = df_web[columns_to_keep]
df_combined_filtered = df_combined[columns_to_keep]

# Concatenate all variations
df_variations = pd.concat([
    df_gold_filtered,
    df_web_filtered, 
    df_combined_filtered
], ignore_index=True)

# Show summary
print("\n--- SUMMARY ---")
print(f"Total samples: {len(df_variations)}")
print(f"Evidence types: {df_variations['evidence_type'].value_counts().to_dict()}")

# Save to JSONL file
output_path = os.path.join(OUTPUT_DIR, "variations.jsonl")
df_variations.to_json(output_path, orient='records', lines=True)
print(f"\nSaved to: {output_path}")

print("\nFirst few rows:")
df_variations.head()

In [ ]:
# Create validation set by claim-level splitting to prevent data leakage
from sklearn.model_selection import train_test_split

# Split claims (not rows) to ensure no leakage between train/val
unique_claims = df_variations.drop_duplicates(subset=['claim'])
train_claims, val_claims = train_test_split(
    unique_claims, test_size=0.10, stratify=unique_claims['label'], random_state=42
)

# Create final datasets with all evidence types for selected claims
train_claim_set = set(train_claims['claim'])
val_claim_set = set(val_claims['claim'])
df_train_final = df_variations[df_variations['claim'].isin(train_claim_set)].reset_index(drop=True)
df_validation = df_variations[df_variations['claim'].isin(val_claim_set)].reset_index(drop=True)

# Validation checks
claim_overlap = set(df_train_final['claim']).intersection(set(df_validation['claim']))
evidence_consistency = all(
    df.groupby('claim')['evidence_type'].nunique().eq(3).all() 
    for df in [df_train_final, df_validation]
)

# Summary
print(f"Dataset split: {len(df_train_final)} train, {len(df_validation)} val")
print(f"Unique claims: {len(train_claims)} train, {len(val_claims)} val")
print(f"Claim leakage: {'none' if len(claim_overlap) == 0 else f'{len(claim_overlap)} overlaps'}")
print(f"Evidence consistency: {'all claims have 3 evidence types' if evidence_consistency else 'some claims are missing evidence types'}")

# Distribution check
for name, df in [("Training", df_train_final), ("Validation", df_validation)]:
    evidence_dist = df['evidence_type'].value_counts(normalize=True).round(3)
    label_dist = df['label'].value_counts(normalize=True).round(3)
    print(f"\n{name}: Evidence {evidence_dist.to_dict()}, Labels {label_dist.to_dict()}")

In [ ]:
# Save the final training and validation sets
# Save datasets split by evidence type
evidence_types = ['gold', 'web_search', 'gold_plus_web_search']

for evidence_type in evidence_types:
    # Filter by evidence type
    train_subset = df_train_final[df_train_final['evidence_type'] == evidence_type]
    val_subset = df_validation[df_validation['evidence_type'] == evidence_type]
    
    # Save training subset
    train_path = os.path.join(OUTPUT_DIR, f"train_{evidence_type}.jsonl")
    train_subset.to_json(train_path, orient="records", lines=True)
    
    # Save validation subset
    val_path = os.path.join(OUTPUT_DIR, f"validation_{evidence_type}.jsonl")
    val_subset.to_json(val_path, orient="records", lines=True)
    
    print(f"{evidence_type.upper()}:")
    print(f"  Training: {len(train_subset)} samples -> {train_path}")
    print(f"  Validation: {len(val_subset)} samples -> {val_path}")

print("\nSummary by evidence type:")
for evidence_type in evidence_types:
    train_count = len(df_train_final[df_train_final['evidence_type'] == evidence_type])
    val_count = len(df_validation[df_validation['evidence_type'] == evidence_type])
    total_count = train_count + val_count
    print(f"  {evidence_type}: {train_count} train + {val_count} val = {total_count} total")

In [ ]:
df_train_final

In [ ]:
# Display validation set shape
print(f"Validation set shape: {df_validation.shape}")
df_validation